In [1]:
import csv
import os
import pymysql

## Config db connection data

In [2]:
DB_HOST = "127.0.0.1"
DB_PORT = 3306
DB_USER = "admin"
DB_PASSWORD = "admin"
DB_NAME = "app_db"

## Setting the output folder

In [3]:
OUTPUT_DIR = "data"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Connecting

In [4]:
connection = pymysql.connect(
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,
    cursorclass=pymysql.cursors.SSCursor,
)

In [5]:
with connection.cursor() as cursor:
    cursor.execute("SHOW FULL TABLES WHERE Table_type = 'BASE TABLE'")
    tables = [row[0] for row in cursor.fetchall()]
    
print(f"Found {len(tables)} tables.")

Found 5 tables.


In [6]:
## Creating csv files

In [7]:
for table in tables:
    quoted_table = "`" + table.replace("`", "``") + "`"
    
    with connection.cursor() as cursor:
        cursor.execute(f"SELECT * FROM {quoted_table}")
    
        columns = [column[0] for column in cursor.description]
        filename = os.path.join(OUTPUT_DIR, f"{table}.csv")
    
        with open(filename, "w", newline="", encoding="utf-8-sig") as file:
            writer = csv.writer(file)
            writer.writerow(columns)
    
            row_count = 0
            while True:
                rows = cursor.fetchmany(1000)
                if not rows:
                    break
                writer.writerows(rows)
                row_count += len(rows)
    
    print(f"Exported {table}: {row_count} rows -> {filename}")
    
    print(f"\nExport completed! CSV files are in: {OUTPUT_DIR}")

Exported classes: 3 rows -> data/classes.csv

Export completed! CSV files are in: data
Exported grades: 3 rows -> data/grades.csv

Export completed! CSV files are in: data
Exported students: 3 rows -> data/students.csv

Export completed! CSV files are in: data
Exported subjects: 3 rows -> data/subjects.csv

Export completed! CSV files are in: data
Exported teacher: 3 rows -> data/teacher.csv

Export completed! CSV files are in: data


In [8]:
connection.close()